In [ ]:
!pip install --upgrade transformers accelerate

In [ ]:
# ── Cell 0: Setup ─────────────────────────────────────────────────────────────
# Phase 2 Activation Extraction — Token Mean-Pooled Residual Stream
#
# Improvement over Phase 1 (last-token): mean-pools the residual stream
# over tokens 50→end for emotion stories. This matches Tim Duffy's approach
# and produces more stable, distributed emotion representations.
#
# Outputs: /kaggle/working/activations_pooled.pkl
#   format: {'resid': {emotion_key: ndarray [n_stories, n_layers, d_model]}}
#   compatible with gemma4-phase1-nrcvad-pca.ipynb (drop-in replacement)
#
# Requires as Kaggle input:
#   - gemma-4-e26b-it model
#   - gemma-4-story-generation notebook output (stories_flat.json)

import os, gc, json, pickle
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# ── Device: TPU → CUDA → CPU ───────────────────────────────────────────────────
try:
    import torch_xla.core.xla_model as xm
    device = xm.xla_device()
    device_type = "tpu"
    print(f"TPU available: {device}")
except (ImportError, RuntimeError):
    xm = None
    if torch.cuda.is_available():
        device = torch.device("cuda")
        device_type = "cuda"
    else:
        device = torch.device("cpu")
        device_type = "cpu"
    print(f"Using device: {device}")

# ── Kaggle input filesystem ────────────────────────────────────────────────────
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


In [ ]:
# ── Cell 1: Paths ─────────────────────────────────────────────────────────────

# ── Model ─────────────────────────────────────────────────────────────────────
MODEL_DIR = "/kaggle/input/models/google/gemma-4/transformers/gemma-4-e26b-it/1"
for candidate in [
    "/kaggle/input/gemma-4-e26b-it",
    "/kaggle/input/gemma4-e26b/transformers/gemma-4-e26b-it/1",
]:
    if not os.path.exists(MODEL_DIR) and os.path.exists(candidate):
        MODEL_DIR = candidate
        break

# ── Emotion stories (from gemma-4-story-generation notebook) ───────────────────
# Attach the story generation notebook output as an input dataset.
STORIES_PATH = None
for candidate in [
    "/kaggle/input/notebooks/bencarson/gemma-4-story-generation-for-174-emotion-concepts/stories_phase2/stories_flat.json",
    "/kaggle/input/gemma-4-story-generation/stories_phase2/stories_flat.json",
    "/kaggle/input/gemma4-story-generation/stories_phase2/stories_flat.json",
    "/kaggle/working/stories_phase2/stories_flat.json",
]:
    if os.path.exists(candidate):
        STORIES_PATH = candidate
        break

if STORIES_PATH is None:
    raise FileNotFoundError(
        "stories_flat.json not found. Attach the gemma-4-story-generation "
        "notebook output as an input dataset."
    )

OUTPUT_PATH = "/kaggle/working/activations_pooled.pkl"

# ── Pooling configuration ──────────────────────────────────────────────────────
# For emotion stories (~150-200 tokens): skip first 50 tokens (instruction
# preamble artifacts) and pool over story content.
# For neutral prompts (~40-80 tokens): pool from token 1 (skip BOS only).
# Both are much more stable than last-token; pool_start is a soft recommendation.
EMOTION_POOL_START = 50
NEUTRAL_POOL_START = 1

print(f"Model dir:    {MODEL_DIR}")
print(f"Stories path: {STORIES_PATH}")
print(f"Output path:  {OUTPUT_PATH}")


In [ ]:
# ── Cell 2: Neutral topics (100) ─────────────────────────────────────────────
# Original 10 (from Phase 1) + 90 new diverse topics.
# Formatted with the chat template to give ~50-80 tokens of context,
# which the model processes as a neutral factual-writing instruction.

NEUTRAL_TOPICS = [
    # Original 10
    "the water cycle and evaporation",
    "how tectonic plates move",
    "the life cycle of a star",
    "how photosynthesis works",
    "the history of the Roman calendar",
    "how semiconductors function",
    "the formation of stalactites",
    "the structure of DNA",
    "how tides are caused by the moon",
    "the process of fermentation",
    # Cooking / Food (10)
    "the Maillard reaction in cooking",
    "how pasta is made from wheat flour",
    "the process of cheese aging and ripening",
    "how bread rises through yeast fermentation",
    "the difference between simmering and boiling",
    "how olive oil is pressed from olives",
    "how salt preserves food from spoilage",
    "the structure and composition of a chicken egg",
    "how coffee beans are decaffeinated",
    "how chocolate is produced from cacao beans",
    # Geography / Earth (10)
    "how river deltas form at coastlines",
    "the formation of desert sand dunes",
    "how glaciers carve valleys and reshape landscapes",
    "how the water table works underground",
    "the formation of coral reefs in tropical oceans",
    "how volcanoes form and create new islands",
    "the jet stream and its effects on weather patterns",
    "how limestone caves form through erosion",
    "the difference between weather and climate",
    "how sedimentary rock layers form over time",
    # History / Technology (10)
    "how the printing press changed information distribution",
    "the construction of Roman aqueducts for water supply",
    "how medieval castles were designed and built",
    "the development of coinage in ancient economies",
    "how the compass changed maritime navigation",
    "the history of paper making in ancient China",
    "how ancient Egyptians preserved mummies",
    "the development of postal systems historically",
    "how the steam engine drove the industrial revolution",
    "the construction of railways in the nineteenth century",
    # Engineering / Technology (10)
    "how transistors amplify electrical signals",
    "the principle behind hydraulic braking systems",
    "how fiber optic cables transmit data using light",
    "how internal combustion engines work",
    "how radio waves are transmitted and received",
    "how mechanical locks and keys work",
    "the structural principles of suspension bridges",
    "how batteries convert chemical energy to electricity",
    "how a microwave oven heats food using radiation",
    "how solar panels convert sunlight to electricity",
    # Biology / Medicine (10)
    "how the immune system identifies and destroys pathogens",
    "the process of cell division in mitosis",
    "how vaccines train the immune system",
    "the structure and function of the human kidney",
    "how bones repair themselves after a fracture",
    "how insulin regulates blood sugar levels",
    "the process of digestion in the stomach and intestines",
    "how the human eye focuses light onto the retina",
    "how antibiotics work against bacterial infections",
    "the structure and function of a cell membrane",
    # Music / Arts (10)
    "how sound waves create musical pitch and tone",
    "the difference between major and minor musical scales",
    "how pigments mix in oil painting",
    "the structure and sections of a symphony orchestra",
    "how string instruments produce sound through resonance",
    "the history of the piano keyboard layout",
    "how architectural acoustics are designed for concert halls",
    "the principles of linear perspective in drawing",
    "how printing techniques evolved in the history of art",
    "how photography captures and reproduces images",
    # Sports / Games (5)
    "how aerodynamics affects a baseball pitch",
    "the rules and piece movements in chess",
    "the physics of a tennis ball bounce on different surfaces",
    "how swimming strokes differ in efficiency",
    "how Olympic swimming pool lanes are standardised",
    # Economics / Finance (10)
    "how compound interest accumulates over time",
    "the role of central banks in managing economies",
    "how supply and demand interact to set prices",
    "how stock exchanges facilitate share trading",
    "how inflation affects purchasing power over time",
    "the difference between GDP and GNP as economic measures",
    "how insurance pools risk across large populations",
    "how currency exchange rates are determined",
    "the concept of opportunity cost in economics",
    "how government bonds work as a form of debt",
    # Languages / Linguistics (5)
    "how related languages share a common ancestor",
    "the difference between syntax and semantics in grammar",
    "how alphabets developed from ancient pictographs",
    "how languages change and evolve across generations",
    "how the International Phonetic Alphabet works",
    # Architecture / Materials (5)
    "how concrete gains structural strength after mixing",
    "the principles of arch bridges and load distribution",
    "how glass is manufactured from silica sand",
    "how skyscrapers resist wind loads through design",
    "how building insulation reduces heat transfer",
    # Mathematics / Logic (5)
    "how prime numbers are distributed among integers",
    "the principles of Euclidean geometry and parallel lines",
    "how binary number systems represent data in computers",
    "the concept of probability in predicting outcomes",
    "how encryption algorithms protect digital information",
]

assert len(NEUTRAL_TOPICS) == 100, f"Expected 100, got {len(NEUTRAL_TOPICS)}"
print(f"Neutral topics: {len(NEUTRAL_TOPICS)}")

NEUTRAL_USER_MSG = (
    "Write a short factual description (3-4 sentences) about {topic}. "
    "Be precise and informative.\n\nDescription:"
)


In [ ]:
# ── Cell 3: Load emotion stories ──────────────────────────────────────────────
with open(STORIES_PATH) as f:
    stories_flat = json.load(f)

# stories_flat: {emotion_key: [story1, story2, ...]} or
#               {emotion_key: {'stories': [...], ...}}  (full metadata version)
# Normalise to flat list.
def _get_stories(v):
    if isinstance(v, list):
        return v
    if isinstance(v, dict) and 'stories' in v:
        return v['stories']
    return []

emotion_stories = {
    k: _get_stories(v)
    for k, v in stories_flat.items()
    if k != '__neutral__'
}

n_emotions = len(emotion_stories)
total_stories = sum(len(v) for v in emotion_stories.values())
print(f"Emotions: {n_emotions}")
print(f"Total stories: {total_stories}")
for k, stories in list(emotion_stories.items())[:3]:
    print(f"  {k}: {len(stories)} stories | first ~80 chars: {stories[0][:80]!r}")


In [ ]:
# ── Cell 4: Load model ────────────────────────────────────────────────────────
print(f"Loading tokenizer from {MODEL_DIR} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, extra_special_tokens={})

print(f"Loading model onto {device_type} ...")
# device_map='auto' is CUDA-only — load to CPU then move.
model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,       # 26B: avoid peak CPU RAM spike
)
model = model.to(device)
model.eval()
print("Model loaded.")

# Infer n_layers from a tiny probe forward pass
_dummy = tokenizer("test", return_tensors="pt").to(device)
with torch.no_grad():
    _out = model(**_dummy, output_hidden_states=True, use_cache=False)
n_layers = len(_out.hidden_states) - 1  # hidden_states[0]=embed; [1..n] = after each block
d_model   = _out.hidden_states[1].shape[-1]
del _dummy, _out; gc.collect()

print(f"n_layers={n_layers}, d_model={d_model}")

if xm is not None:
    xm.mark_step()


In [ ]:
# ── Cell 5: Mean-pooled activation extraction ─────────────────────────────────
# Returns [n_layers, d_model] float32: mean over token positions [pool_start:]
# at each residual stream layer.
#
# hidden_states[0]   = token embedding (before any transformer block)
# hidden_states[i+1] = residual stream after block i (hook_resid_post equiv.)
#
# pool_start: first token index to include in the mean.
#   Emotion stories: 50 — skips prompt-preamble artefacts, pools story content.
#   Neutral prompts: 1  — prompts are short (~40-80 tok); skip BOS only.

def extract_mean_pool_acts(text, model, tokenizer, n_layers, pool_start=50):
    inputs = tokenizer(
        text, return_tensors="pt", truncation=True, max_length=2048
    ).to(device)
    seq_len = inputs["input_ids"].shape[1]

    # Clamp pool_start so we always average at least 1 token
    start = min(pool_start, seq_len - 1)

    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True, use_cache=False)

    if xm is not None:
        xm.mark_step()

    acts = np.stack([
        outputs.hidden_states[i + 1][0, start:, :].float().mean(dim=0).cpu().numpy()
        for i in range(n_layers)
    ])  # [n_layers, d_model]
    return acts


In [ ]:
# ── Cell 6: Extract neutral activations ──────────────────────────────────────
# 100 topics × 1 text each → [100, n_layers, d_model]
#
# Each neutral text is the chat-formatted instruction asking for a factual
# description. The model's residual stream while reading a neutral instruction
# is the baseline we need — no generation required.
# Pool from token 1+ (skip BOS; prompts are ~40-80 tokens total).

neutral_acts_list = []
for i, topic in enumerate(NEUTRAL_TOPICS):
    msg = NEUTRAL_USER_MSG.format(topic=topic)
    chat_text = tokenizer.apply_chat_template(
        [{"role": "user", "content": msg}],
        add_generation_prompt=True,
        tokenize=False,
    )
    acts = extract_mean_pool_acts(
        chat_text, model, tokenizer, n_layers, pool_start=NEUTRAL_POOL_START
    )
    neutral_acts_list.append(acts)
    if i % 20 == 0:
        tok_len = tokenizer(chat_text, return_tensors="pt").input_ids.shape[1]
        print(f"  [{i:3d}] {topic[:45]:45s}  tok_len={tok_len}")
    gc.collect()

neutral_acts = np.stack(neutral_acts_list)  # [100, n_layers, d_model]
print(f"\nNeutral activations: {neutral_acts.shape}")

# Sanity: per-layer norms should be consistent across topics
norms = np.linalg.norm(neutral_acts[:, n_layers // 2, :], axis=-1)
print(f"Mid-layer norms: min={norms.min():.1f}  max={norms.max():.1f}  mean={norms.mean():.1f}")


In [ ]:
# ── Cell 7: Extract emotion activations ──────────────────────────────────────
# For each emotion: [n_stories, n_layers, d_model]
# Emotion stories are ~150-200 tokens. Pool from token 50+ to skip any
# early-generation artefacts and capture distributed story content.

resid_acts = {}
resid_acts["__neutral__"] = neutral_acts   # store neutral under the Phase-1 key

emotion_keys = sorted(emotion_stories.keys())
n_emotions = len(emotion_keys)

for ei, emotion in enumerate(emotion_keys):
    stories = emotion_stories[emotion]
    story_acts = []
    for text in stories:
        acts = extract_mean_pool_acts(
            text, model, tokenizer, n_layers, pool_start=EMOTION_POOL_START
        )
        story_acts.append(acts)
    resid_acts[emotion] = np.stack(story_acts)  # [n_stories, n_layers, d_model]
    if ei % 20 == 0 or ei == n_emotions - 1:
        print(f"  [{ei+1:3d}/{n_emotions}] {emotion}  shape={resid_acts[emotion].shape}")
    gc.collect()

print(f"\nAll emotions extracted. Keys: {len(resid_acts)} (incl. __neutral__)")


In [ ]:
# ── Cell 8: Save ──────────────────────────────────────────────────────────────
# Drop-in replacement for Phase 1 activations.pkl.
# PCA notebook (gemma4-phase1-nrcvad-pca.ipynb) reads:
#   saved['resid'][emotion_key]  → ndarray [n_stories, n_layers, d_model]

with open(OUTPUT_PATH, "wb") as f:
    pickle.dump(
        {
            "resid": resid_acts,
            "meta": {
                "extraction": "mean_pool",
                "emotion_pool_start": EMOTION_POOL_START,
                "neutral_pool_start": NEUTRAL_POOL_START,
                "n_layers": n_layers,
                "d_model":  d_model,
                "n_emotions": len(emotion_keys),
                "n_neutral": len(NEUTRAL_TOPICS),
            },
        },
        f,
    )
print(f"Saved to {OUTPUT_PATH}")
print(f"  Emotions: {len(emotion_keys)}")
print(f"  Neutral:  {len(NEUTRAL_TOPICS)}")
print(f"  Sample shape: {resid_acts[emotion_keys[0]].shape}")


In [ ]:
# ── Cell 9: Verify — quick layer-sweep correlation ────────────────────────────
# Spot-check: does valence PC1 correlate better at some layers than others?
# Uses the 171-emotion NRC-VAD match from the PCA notebook logic.
from sklearn.decomposition import PCA
from scipy import stats
import pandas as pd

NRC_PATH = "/kaggle/input/datasets/manjitbaishya2026/nrc-vad/NRC-VAD-Lexicon-v2.1.txt"
if not os.path.exists(NRC_PATH):
    print("NRC-VAD lexicon not found — skipping correlation check.")
else:
    nrc = pd.read_csv(NRC_PATH, sep="\t", header=0,
                      names=["word", "valence", "arousal", "dominance"])
    nrc = nrc.drop_duplicates("word").set_index("word")

    def lookup_vad(name):
        word = name.replace("_", " ").lower()
        if word in nrc.index:
            return nrc.loc[word, ["valence", "arousal", "dominance"]].values.astype(float)
        first = word.split()[0]
        if first in nrc.index:
            return nrc.loc[first, ["valence", "arousal", "dominance"]].values.astype(float)
        return None

    # Build mean direction matrix at each layer
    matched_emotions = [e for e in emotion_keys if lookup_vad(e) is not None]
    vad = np.array([lookup_vad(e) for e in matched_emotions])
    print(f"Matched {len(matched_emotions)} emotions to NRC-VAD")

    LAYERS_TO_CHECK = list(range(0, n_layers, max(1, n_layers // 10)))
    print(f"\nLayer sweep (valence PC1 correlation):")
    print(f"{'Layer':>6}  {'valence_r':>10}  {'arousal_r':>10}")

    neutral_mean_all = neutral_acts.mean(axis=0)  # [n_layers, d_model]

    for layer in LAYERS_TO_CHECK:
        dirs = np.array([
            resid_acts[e][:, layer, :].mean(axis=0) - neutral_mean_all[layer]
            for e in matched_emotions
        ])
        pca = PCA(n_components=1)
        scores = pca.fit_transform(dirs)
        rv, _ = stats.pearsonr(scores[:, 0], vad[:, 0])
        ra, _ = stats.pearsonr(scores[:, 0], vad[:, 1])
        print(f"{layer:>6}  {rv:>+10.3f}  {ra:>+10.3f}")
